In [1]:
from pyMoove import io, plotting, calculations, cleaning, statistics, interaction_detection

from scripts.zone_info import get_zone_info
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
######################################
# DATA LOADING AND CLEANING PIPLINE   #
####################################


# Resolve data directory (works from project root or scripts directory)
DATA_DIR = Path("data")

data = io.load_processed_dataset(DATA_DIR)
zones_path = DATA_DIR / "zone_info.csv"
zones = get_zone_info(zones_path)

# Calculate statistics for all feeding bins
#bins = sorted(data["feeding"]["bin"].dropna().unique())
#bin_statistics = calculations.calculate_bin_position_statistics(data, bins, outlier_confidence=0.9999)
#bin_statistics.to_parquet(bin_statistics_path, index=False)

# Save results to Parquet
bin_statistics_path = DATA_DIR / "bin_position_stats.parquet"
bin_statistics = io.load_bin_stats_parquet(bin_statistics_path)

# Data cleaning that removes all replacement events where there is only positional data on one of the cows
replacements = data['replacements']
stats = statistics.inspect_replacement_event_statistics(data['positions'], replacements, bin_statistics, time_before=30, time_after=30)
replacements = cleaning.filter_replacement_events(replacements, stats)
stats = statistics.inspect_replacement_event_statistics(data['positions'], replacements, bin_statistics, time_before=30, time_after=30)

# Remove outliers. Parameters has been set based on based on manual inspection of outlier data. 
outliers_mask = (stats['actor_bin_min_m'] > 1) | (stats['reactor_bin_min_m'] > 1.11) | (stats['cow_min_m'] > 2.136)
replacements = replacements[~outliers_mask]
print(f"\nTotal replacement events after removing outliers: {len(replacements)}\n")
stats = statistics.inspect_replacement_event_statistics(data['positions'], replacements, bin_statistics, time_before=30, time_after=30)




Loaded dataset in folder: data
  positions Memory: 571.74 MiB
  feeding Memory: 0.28 MiB
  replacements Memory: 0.02 MiB
Total memory: 572.05 MiB

Total event before cleaning: 550
Events retained: 470
Expected removal: 80
Events removed: 80

Total replacement events after removing outliers: 431



In [ ]:
###########################
# BIN INTERACTION PIPELINE #
###########################


# Detection of gos events
bins = range(1, 11)
# We filter out the specific bins we want to look at
bin_replacements = replacements[replacements['bin'].isin(bins)]
bin_interactions = interaction_detection.detect_bin_interactions(data['positions'], bin_statistics, bins, max_bin_distance=1.5)
# This is needed to ensure non duplicate indices, hopefully Oscar can fix this
bin_interactions = bin_interactions.reset_index(drop=True)

#Statistics about how many interactions and replacements there are as well as how many of the replacements were identified as bin_interaction events (recall)
recall = interaction_detection.check_bin_interactions(bin_replacements, bin_interactions, acceptance_interval=30, cow_role_matters=False, show_detections=False)

# Validation of which gos events are actual replacements or not
# The output is a mask of what bin_interactions are real replacements:
# bin_interactions[label_mask]          - bin interactions that correspond to replacement events
# bin_interactions[~label_mask]         - bin interactions that do NOT correspond to replacement events
label_mask = interaction_detection.validate_bin_interactions(bin_replacements, bin_interactions, acceptance_interval=30)

Actual replacements: 61
Bin interactions: 79961
57 / 61 replacement events were detected

Detected replacements: 79961
Actual replacements: 61
Detected replacements labeled as real: 63
Detected replacements labeled as not real: 79898



In [ ]:
bin_interactions[label_mask].sort_values(['bin', 'time']).head(24)

In [ ]:
bin_replacements.sort_values(['bin', 'time']).head(24)

In [ ]:
# Looking at the results we might find outliers and would like to inspect these
#columns = ["cow_min_m"]
#columns = ["actor_bin_min_m"]
columns = ["reactor_bin_min_m"]

# If you want to select several outliers combine them uniqulie into one dataframe
# Otherwise find_boxplot_outliers() returns all outlier events for one specific column
outliers = []
for column in columns:
    outliers.append(statistics.find_boxplot_outliers(stats, column))

# All unique event outliers are collected into one dataframe.
unique_outliers = pd.concat(outliers).drop_duplicates(subset="event_id")

# We can inspect the outliers however we like
print(f"Total replacement events: {len(stats)}")
print(f"Unique outlier events: {len(unique_outliers)}")

print(unique_outliers.sort_values(columns, ascending=False).head(34)[column])


In [ ]:
bins = range(1, 11)
bin_replacements = replacements[replacements['bin'].isin(bins)]
#bin_replacements.sort_values('time').reset_index(drop=True)
bin_interactions = interactions_detection.detect_replacements(data['positions'], bin_statistics, bins, max_bin_distance=2.0)
recall = interactions_detection.check_replacement_recall(bin_replacements, detected_replacements, acceptance_interval=60, cow_role_matters=False, show_detections=False)

In [ ]:
replacements.head(10)

In [ ]:
##############
# BOX PLOTS
##############
time_before = 30
time_after = 30
repl_stats_dir = Path("Replacement_statistics")
repl_stats_path = DATA_DIR / repl_stats_dir
repl_stats_path.mkdir(parents=True, exist_ok=True)
name = 'filtered_replacements'

def statistics_box_plots():
    # Cow to cow distances
    fig, ax = plt.subplots()
    stats[["cow_min_m", "cow_mean_m", "cow_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Cow to cow distances during replacement events",
    )
    filepath = repl_stats_path / f"cow_to_cow_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

    # Actor to bin distances
    fig, ax = plt.subplots()
    stats[["actor_bin_min_m", "actor_bin_mean_m", "actor_bin_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Actor to feed bin distances during replacement events",
    )
    filepath = repl_stats_path / f"actor_to_bin_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

    # Reactor to bin distances
    fig, ax = plt.subplots()
    stats[["reactor_bin_min_m", "reactor_bin_mean_m", "reactor_bin_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Reactor to feed bin distances during replacement events",
    )
    filepath = repl_stats_path / f"reactor_to_bin_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

plt.show()

In [ ]:
##############
# BOX PLOTS
##############
time_before = 30
time_after = 30
repl_stats_dir = Path("Replacement_statistics")
repl_stats_path = DATA_DIR / repl_stats_dir
repl_stats_path.mkdir(parents=True, exist_ok=True)
name = 'detected_replacements'

detection_events_stats = statistics.inspect_replacement_event_statistics(data['positions'], detected_replacements, bin_statistics, time_before=30, time_after=30)

if True:
    # Cow to cow distances
    fig, ax = plt.subplots()
    detection_events_stats[["cow_min_m", "cow_mean_m", "cow_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Cow to cow distances during replacement events",
    )
    filepath = repl_stats_path / f"cow_to_cow_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

    # Actor to bin distances
    fig, ax = plt.subplots()
    detection_events_stats[["actor_bin_min_m", "actor_bin_mean_m", "actor_bin_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Actor to feed bin distances during replacement events",
    )
    filepath = repl_stats_path / f"actor_to_bin_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

    # Reactor to bin distances
    fig, ax = plt.subplots()
    detection_events_stats[["reactor_bin_min_m", "reactor_bin_mean_m", "reactor_bin_max_m"]].plot.box(
        ax=ax,
        ylabel="Distance [m]",
        title=f"Reactor to feed bin distances during replacement events",
    )
    filepath = repl_stats_path / f"reactor_to_bin_boxplot_{name}.png"
    fig.savefig(filepath, dpi=300)

plt.show()

In [ ]:
animations = []
for i in [345, 357]:
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:
animations = []
for i in range(0, 4):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], bin_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:

animations = []
for i in range(1265, 1270):
    fig, ax, animation = plotting.plot_replacement_example(data['positions'], detected_replacements, zones=zones, bin_statistics=bin_statistics, event_index=i, animate=True, relative_start_time=30, relative_end_time=30)
    animations.append(animation)

In [ ]:
bins = range(1,59)
plotting.plot_bins(zones, bin_statistics, bins)

In [ ]:
fig, ax = plotting.plot_feeding_event(data, zones, event_index=0, bin_statistics=bin_statistics)

In [ ]:

# Option A: Plot all feeding events for a specific bin (e.g. Bin 21, 24, etc.)
bins = range(58, 59)
for i in bins:
    target_bin = i
    fig, ax = plotting.plot_bin_feeding_positions(data, zones, bin_id=target_bin, color_by_cow=True, outlier_confidence=0.9999)

# Option B: Plot a single feeding event by index (uncomment to use):
# fig, ax = plot_feeding_event(data, zones, event_index=0)